In [22]:
import pandas as pd
import yfinance as yf

In [23]:
# Create a list of tickers
tickers = pd.read_csv("../data/processed/stocks_transformed.csv")['ticker'].tolist()
tickers

['MP',
 'USAR',
 'LYC',
 'UUUU',
 'ALB',
 'SQM',
 '002460',
 '3402',
 'HXL',
 '4188',
 'MMM',
 'DD',
 'BAS',
 '1COV',
 'EVK',
 'MT',
 'NVDA',
 'GOOGL',
 'META',
 '002230',
 'U',
 'GOOGL',
 'QCOM',
 'AMD',
 'INTC',
 'AMBA',
 'LSCC',
 'CEVA',
 'NXPI',
 'STM',
 'IFX',
 'TXN',
 '2454',
 'MPWR',
 '300750',
 '373220',
 '006400',
 '6752',
 '002594',
 '300014',
 '6981',
 '6762',
 'NVTS',
 'WOLF',
 'ADI',
 'SONY',
 '005930',
 'HSAI',
 'OUST',
 'LAZR',
 'NOVT',
 '2498',
 'ALGM',
 'VPG',
 'CGNX',
 '688322',
 '603501',
 '6324',
 '688017',
 '6268',
 'SHA',
 '6481',
 '601100',
 '6594',
 'RRX',
 'MOGA',
 '603728',
 '002979',
 '688698',
 'ALNT',
 'RBC',
 'AME',
 'RRX',
 '002050',
 '601689',
 '003021',
 '300124',
 'TSLA',
 '005380',
 '688836',
 '9880',
 '1810',
 'XPEV',
 'SIE',
 'ABBN',
 '000333',
 '6954',
 '6506',
 '2317',
 'ROK',
 'JBL',
 'AMZN',
 'BMW',
 'MBG',
 '005380',
 'GXO',
 'DHL',
 'WMT',
 'TM',
 'MELI',
 'PTC',
 'NOW',
 'TMV',
 'TRMB',
 '018260']

Some tickers are not recognized by yfinance (usually non-US stocks) and need to be modified to add the country suffix at the end of the ticker. In order to optimize the tests, we use patterns to try to deduce ASAP the quotation country.

In [24]:
# 1. Read CSV while forcing the ticker column to stay as text (keeps leading zeros)
df = pd.read_csv("../data/processed/stocks_transformed.csv", dtype={"ticker": str})
tickers = df["ticker"].dropna().tolist()

In [25]:
def find_valid_ticker(raw_ticker):
    
    # 2. Find the tickers working with yfinance library.
    ticker = str(raw_ticker).strip().upper()
    candidates = []

    # Check numeric-only tickers based on digit length
    if ticker.isdigit():
        digit_count = len(ticker)

        if digit_count == 4:
            # 4-digit tickers: Try 4-digit markets first (Japan, Taiwan, Hong Kong)
            ticker_4 = ticker.zfill(4)
            candidates.append(ticker_4 + ".T")   # Japan
            candidates.append(ticker_4 + ".TW")  # Taiwan TWSE
            candidates.append(ticker_4 + ".TWO") # Taiwan TPEx
            candidates.append(ticker_4 + ".HK")  # Hong Kong

            # Fallback: Zero-pad to 6 digits for Korea / China
            ticker_6 = ticker.zfill(6)
            candidates.append(ticker_6 + ".KS")  # Korea KOSPI
            candidates.append(ticker_6 + ".KQ")  # Korea KOSDAQ
            candidates.append(ticker_6 + ".SS")  # Shanghai
            candidates.append(ticker_6 + ".SZ")  # Shenzhen

        elif digit_count == 6:
            # 6-digit tickers: Try 6-digit markets FIRST (Korea, China)
            candidates.append(ticker + ".KS")  # Korea KOSPI
            candidates.append(ticker + ".KQ")  # Korea KOSDAQ
            candidates.append(ticker + ".SS")  # Shanghai
            candidates.append(ticker + ".SZ")  # Shenzhen

            # Fallback: Trim to 4 digits for Japan / Taiwan
            ticker_4 = ticker[-4:]
            candidates.append(ticker_4 + ".T")   # Japan
            candidates.append(ticker_4 + ".TW")  # Taiwan
            candidates.append(ticker_4 + ".HK")  # Hong Kong

        else:
            # 5-digit or other lengths (e.g. Hong Kong 1810 / 9880)
            ticker_4 = ticker.zfill(4)
            ticker_6 = ticker.zfill(6)
            candidates.append(ticker_4 + ".HK")
            candidates.append(ticker_6 + ".KS")
            candidates.append(ticker_6 + ".SS")
            candidates.append(ticker_6 + ".SZ")

    else:
        # Non-numeric tickers (US & popular European tickers)
        candidates.append(ticker)             # US
        candidates.append(ticker + ".DE")     # Germany
        candidates.append(ticker + ".F")      # Frankfurt
        candidates.append(ticker + ".L")      # London
        candidates.append(ticker + ".PA")     # Paris
        candidates.append(ticker + ".SW")     # Switzerland

    # Remove duplicates while keeping exact order of priority
    candidates = list(dict.fromkeys(candidates))

    # Test candidate tickers to see if a yfinance command works (valid ticker):
    for candidate in candidates:
        try:
            hist = yf.Ticker(candidate).history(period="1d")
            if not hist.empty:
                print(f"SUCCESS: {raw_ticker} -> {candidate}")
                return candidate
        except Exception:
            pass

    print(f"NOT FOUND: {raw_ticker}")


In [26]:
# 2. run over all tickers and collect results
results = {}
not_found_tickers = []

for t in tickers:
    matched = find_valid_ticker(t)
    results[t] = matched
    
    # Collect missing tickers into a list
    if matched is None:
        not_found_tickers.append(t)

print("Finished processing all tickers.")

$LYC: possibly delisted; no price data found  (period=1d)


SUCCESS: MP -> MP
SUCCESS: USAR -> USAR


$LYC.DE: No data found, symbol may be delisted


SUCCESS: LYC -> LYC.F
SUCCESS: UUUU -> UUUU
SUCCESS: ALB -> ALB
SUCCESS: SQM -> SQM
SUCCESS: 002460 -> 002460.KS
SUCCESS: 3402 -> 3402.T
SUCCESS: HXL -> HXL
SUCCESS: 4188 -> 4188.T
SUCCESS: MMM -> MMM
SUCCESS: DD -> DD


$BAS: No data found, symbol may be delisted


...

SUCCESS: 000333 -> 000333.SZ
SUCCESS: 6954 -> 6954.T
SUCCESS: 6506 -> 6506.T
SUCCESS: 2317 -> 2317.T
SUCCESS: ROK -> ROK
SUCCESS: JBL -> JBL


$BMW: possibly delisted; no price data found  (period=1d)


SUCCESS: AMZN -> AMZN
SUCCESS: BMW -> BMW.DE


$MBG: No data found, symbol may be delisted
$DHL: possibly delisted; no price data found  (period=1d)


SUCCESS: MBG -> MBG.DE
SUCCESS: 005380 -> 005380.KS
SUCCESS: GXO -> GXO
SUCCESS: DHL -> DHL.DE
SUCCESS: WMT -> WMT
SUCCESS: TM -> TM
SUCCESS: MELI -> MELI
SUCCESS: PTC -> PTC
SUCCESS: NOW -> NOW
SUCCESS: TMV -> TMV
SUCCESS: TRMB -> TRMB
SUCCESS: 018260 -> 018260.KS
Finished processing all tickers.


In [27]:
# 3. Print the missing tickers summary at the end
print(f"SUMMARY: {len(not_found_tickers)} ticker(s) could not be resolved:")
if not_found_tickers:
    print(not_found_tickers)
else:
    print("All tickers were successfully matched!")

SUMMARY: 2 ticker(s) could not be resolved:
['SHA', 'MOGA']


In [30]:
# 4. Update ticker column with resolved tickers
df["ticker"] = df["ticker"].replace(results)

# Remove rows where no valid ticker was found
df = df.dropna(subset=["ticker"])
df

,ticker,layer_id
0,MP,['1']
1,USAR,['1']
2,LYC.F,['1']
3,UUUU,['1']
4,ALB,['1']
...,...,...
100,PTC,['13']
101,NOW,['13']
102,TMV,['13']
103,TRMB,['13']


# Export

In [32]:
# Export
df.to_csv("../data/processed/stocks_transformed_verified.csv", index=False)